# HuggingFace vs vLLM: Tokens/sec Benchmark

Loads a ~300MB model (`distilgpt2`) and compares generation throughput (tokens/second)
between plain HuggingFace `transformers` and `vLLM`.

**GPU target: NVIDIA T4.**
- T4 does **not** support `bfloat16` natively -> we force `float16` everywhere.
- T4 does **not** support FlashAttention-2 (Ampere+ only) -> we avoid `attn_implementation="flash_attention_2"`.
- vLLM is configured with `dtype="float16"` and `enforce_eager=True` for safer compatibility on T4 (avoids some CUDA-graph/kernel edge cases on older architectures).
- vLLM requires Linux (native or WSL2). It does not run natively on Windows.

## 1. Install dependencies

Run this cell once. If you're on Windows, run this notebook inside WSL2 (Ubuntu) with CUDA drivers set up for the T4, since vLLM does not support native Windows.

In [ ]:
%pip install -q --upgrade pip
%pip install -q "transformers>=4.42" accelerate pandas matplotlib
# vLLM: pin a version known to work with T4 (Turing, compute capability 7.5)
%pip install -q "vllm>=0.5.0"

In [ ]:
import time
import torch

MODEL_ID = "distilgpt2"  # ~330MB fp32 checkpoint, small enough to be a quick benchmark

assert torch.cuda.is_available(), "CUDA GPU not available. This notebook requires a T4 GPU."

device_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f"Detected GPU: {device_name} (compute capability {major}.{minor})")

if "T4" not in device_name:
    print("WARNING: This notebook is tuned for T4 GPUs. Detected a different GPU; "
          "results/settings below (float16, enforce_eager) are still safe but may not be optimal.")

# T4 = Turing architecture (sm_75): no native bfloat16, no FlashAttention-2.
DTYPE = torch.float16
print(f"Using dtype: {DTYPE}")

## 2. Shared benchmark config

In [ ]:
PROMPT = "The future of artificial intelligence is"
NUM_OUTPUT_TOKENS = 128
NUM_RUNS = 3          # repeat and average to reduce noise
BATCH_SIZE = 8        # concurrent sequences, to also show batched throughput

prompts = [PROMPT] * BATCH_SIZE

## 3. HuggingFace Transformers baseline

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,           # float16 for T4
    attn_implementation="eager", # avoid flash-attention path, unsupported on T4
).to("cuda")
hf_model.eval()
print("HF model loaded.")

In [ ]:
def run_hf_benchmark():
    inputs = hf_tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

    # warmup
    with torch.no_grad():
        hf_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=hf_tokenizer.pad_token_id,
        )
    torch.cuda.synchronize()

    durations = []
    total_new_tokens = 0
    for _ in range(NUM_RUNS):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            out = hf_model.generate(
                **inputs,
                max_new_tokens=NUM_OUTPUT_TOKENS,
                do_sample=False,
                pad_token_id=hf_tokenizer.pad_token_id,
            )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        durations.append(elapsed)

        new_tokens_per_seq = out.shape[1] - inputs["input_ids"].shape[1]
        total_new_tokens += new_tokens_per_seq * out.shape[0]

    avg_duration = sum(durations) / len(durations)
    avg_new_tokens_per_run = total_new_tokens / NUM_RUNS
    tokens_per_sec = avg_new_tokens_per_run / avg_duration
    return {
        "framework": "HuggingFace Transformers",
        "avg_duration_sec": avg_duration,
        "avg_new_tokens": avg_new_tokens_per_run,
        "tokens_per_sec": tokens_per_sec,
    }

hf_result = run_hf_benchmark()
hf_result

In [ ]:
# Free HF model before loading vLLM to avoid competing for T4 VRAM (T4 has 16GB)
del hf_model
torch.cuda.empty_cache()

## 4. vLLM benchmark

Note: vLLM must run on Linux (native or WSL2). `enforce_eager=True` disables CUDA graph capture,
which is a safer default on T4/older GPUs. `dtype="float16"` matches T4's supported precision.

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_ID,
    dtype="float16",          # T4-compatible precision
    enforce_eager=True,       # avoid CUDA graph issues on older (Turing) GPUs
    gpu_memory_utilization=0.85,
    max_model_len=512,
)
print("vLLM engine loaded.")

In [ ]:
def run_vllm_benchmark():
    sampling_params = SamplingParams(
        max_tokens=NUM_OUTPUT_TOKENS,
        temperature=0.0,  # greedy, matches HF do_sample=False
    )

    # warmup
    llm.generate(prompts, SamplingParams(max_tokens=8, temperature=0.0))

    durations = []
    total_new_tokens = 0
    for _ in range(NUM_RUNS):
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling_params)
        elapsed = time.perf_counter() - start
        durations.append(elapsed)

        total_new_tokens += sum(len(o.outputs[0].token_ids) for o in outputs)

    avg_duration = sum(durations) / len(durations)
    avg_new_tokens_per_run = total_new_tokens / NUM_RUNS
    tokens_per_sec = avg_new_tokens_per_run / avg_duration
    return {
        "framework": "vLLM",
        "avg_duration_sec": avg_duration,
        "avg_new_tokens": avg_new_tokens_per_run,
        "tokens_per_sec": tokens_per_sec,
    }

vllm_result = run_vllm_benchmark()
vllm_result

## 5. Compare results

In [ ]:
import pandas as pd

df = pd.DataFrame([hf_result, vllm_result])
df["speedup_vs_hf"] = df["tokens_per_sec"] / hf_result["tokens_per_sec"]
df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(df["framework"], df["tokens_per_sec"], color=["#4C72B0", "#55A868"])
plt.ylabel("Tokens / second")
plt.title(f"{MODEL_ID} throughput on T4 GPU (batch={BATCH_SIZE}, dtype=float16)")
for i, v in enumerate(df["tokens_per_sec"]):
    plt.text(i, v, f"{v:.1f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()